# Survival Modeling

Objective: modeling the survivability of the patients using the clinical dataset.

## Importing Modules

In [ ]:
import pandas as pd

# model selection
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV

# my module
from leukemia.models import evaluation
from leukemia.data import loader
from leukemia.models import evaluation

# preprocessing and pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import  make_pipeline

#LGBM
from lightgbm import LGBMRegressor

# Cox model
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import as_concordance_index_ipcw_scorer




## Importing data

In [ ]:
X, y = loader.split_clinical_processed("../data/processed/clinical_train_processed_v1.csv")

num_cols = ['BM_BLAST_LOG', 'WBC_LOG', 'ANC_LOG', 'MONOCYTES_LOG', 'HB',
       'PLT_LOG']
binary_cols = ['BM_BLAST_MISSING', 'WBC_MISSING',
       'ANC_MISSING','MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING',
       'CYTO_NORMAL', 'CYTO_COMPLEX', 'CYTO_ABNORMAL', 'CYTO_MISSING']

## Splitting data training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## 1. Baseline: LightGBM

In [ ]:
preprocess_LGB = ColumnTransformer([
    ("num", "passthrough", num_cols),("binary", "passthrough", binary_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER']),
])

mod_pipe_LGB = LGBMRegressor()

pipe_LGB = make_pipeline(preprocess_LGB, mod_pipe_LGB)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scorer = evaluation.ipcw_c_index
scores = []

for fold, (train_index, val_index) in enumerate(cv.split(X_train, y_train["OS_STATUS"])):
    X_train_cv, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train_cv, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

    pipe_LGB.fit(X_train_cv, y_train_cv['OS_YEARS'])

    y_train_cv = loader.make_surv_target(y_train_cv['OS_YEARS'], y_train_cv['OS_STATUS'])
    y_val = loader.make_surv_target(y_val['OS_YEARS'], y_val['OS_STATUS'])

    scores.append({"fold": fold, "train_sore": scorer(y_train_cv, y_train_cv, -pipe_LGB.predict(X_train_cv)),
    "val_sore": scorer(y_train_cv, y_val, -pipe_LGB.predict(X_val))})

scores = pd.DataFrame(scores)
print(scores, scores["val_sore"].mean(), scores["val_sore"].std())

#pipe_LGB.fit(X_test, y_test["OS_YEARS"])

#y_train_scorer = loader.make_surv_target(y_train['OS_YEARS'], y_train['OS_STATUS'])
#y_test_scorer = loader.make_surv_target(y_test['OS_YEARS'], y_test['OS_STATUS'])

#print(f"Test score: {scorer(y_train_scorer, y_test_scorer, -pipe_LGB.predict(X_test))}")

The LGBM does not take the censoring into account and therefore consider a deceased patient at 2 year as a censored patient at 2 years.

## 2. First COX regression model

The protocol require 2 adjustments :
- the StratifiedKFold must stratify on OS_STATUS
- the c-index must have OS_STATUS from y_train
We are developing a method to adapt the protocol.

In [ ]:
def run_cv_cox(pipeline, X_train: pd.DataFrame, y_train: pd.DataFrame, cv: StratifiedKFold, scorer) -> tuple :
    """
    Evaluate the cox model
    :param model: model we want to evaluate
    :param X: feature dataset
    :param y: target dataset
    :param cv: cross validation split
    :param scorer: scoring function: IPCW C-index
    :return: the cv results (n°fold, train_sore, val_sore) and the mean score of the validation sets
    """
    scores = []
    for fold, (train_index, val_index) in enumerate(cv.split(X_train, y_train["OS_STATUS"])):
        X_train_cv, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_cv, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

        y_train_cv = loader.make_surv_target(y_train_cv['OS_YEARS'], y_train_cv['OS_STATUS'])
        y_val = loader.make_surv_target(y_val['OS_YEARS'], y_val['OS_STATUS'])

        pipeline.fit(X_train_cv, y_train_cv)

        scores.append({"fold": fold, "train_sore": scorer(y_train_cv, y_train_cv, pipeline.predict(X_train_cv)),
        "val_sore": scorer(y_train_cv, y_val, pipeline.predict(X_val))})
        score_mean = {'mean': pd.DataFrame(scores)["val_sore"].mean(), 'std': pd.DataFrame(scores)["val_sore"].std()}

    return pd.DataFrame(scores), pd.DataFrame([score_mean])

First Cox regression: we are using the base prprocessed data set with:
- biological parameters transformed with log1p
- biological parameters missing values indicators
- cytogenetics split into 4 features:
    - normal
    - abnormal
    - missing
    - complex

We are crating a preprocess_base transformer:
- Standard scale the numerical features
- Replacing the missing values in the numerical features by the median
- One hot encoding the discrete feature CENTER


In [ ]:
preprocess_base = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols),("binary", "passthrough", binary_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER'])])

cv= evaluation.get_cv(5)
scorer = evaluation.ipcw_c_index
pipe_cox_base = make_pipeline(preprocess_base, CoxPHSurvivalAnalysis(alpha=1e-4))
fold_scores, mean_score = run_cv_cox(pipe_cox_base, X_train, y_train, cv, scorer)
print(fold_scores)
print(mean_score)

Our reference score for the cox Regression is 0.684738 ± 0.027847. \
The score of the cox regression is higher than our BM_BLAST baseline (0.616 ± 0.009) and the LGBM baseline (0.599664 ± 0.022042).

## 3. Cox Regression on the features engineered CYTOGENETICS

### Importing the dataset

In [ ]:
X, y = loader.split_clinical_processed('../data/processed/clinical_train_feature_engin.csv')
num_cols = ['BM_BLAST_LOG', 'WBC_LOG', 'ANC_LOG', 'MONOCYTES_LOG', 'HB',
       'PLT_LOG']
binary_cols = ['BM_BLAST_MISSING', 'WBC_MISSING',
       'ANC_MISSING','MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

We are crating a preprocess_v1 transformer:
- Standard scale the numerical features
- Replacing the missing values in the numerical features by the median
- One hot encoding the discrete features : CENTER and CYTO_LVL

In [ ]:
preprocess_v1 = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("binary", "passthrough", binary_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER', 'CYTO_LVL'])])

pipe_cox_v1 = make_pipeline(preprocess_v1, CoxPHSurvivalAnalysis(alpha=1e-4))

fold_scores, mean_score = run_cv_cox(pipe_cox_v1, X_train, y_train, cv, scorer)
print(fold_scores)
print(mean_score)

In [ ]:
coefs_pipe_cox_v1 = pd.Series(pipe_cox_v1[-1].coef_, index=pipe_cox_v1[0].get_feature_names_out()).sort_values()
coefs_pipe_cox_v1

Unfortunately the featured enginered version shows no real difference compared to the base version. The c-index is 0.690717 ± 0.025865. \
The coefficients of the models show low values for the missing indicators, suggesting a low impact on the prediction.
As expected Cyto_LVL very poor and poor have a positive impact on the prediction.


## 4. Pruning the features

We are keeping the feature engineered dataset, but we are removing the missing features indicators.

## Importing data

In [ ]:
X, y = loader.split_clinical_processed('../data/processed/clinical_train_feature_engin.csv')

X.drop(['BM_BLAST_MISSING', 'WBC_MISSING', 'ANC_MISSING', 'MONOCYTES_MISSING', 'HB_MISSING', 'PLT_MISSING'],axis=1, inplace=True)

num_cols = ['BM_BLAST_LOG', 'WBC_LOG', 'ANC_LOG', 'MONOCYTES_LOG', 'HB',
       'PLT_LOG']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
preprocess_v1 = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER', 'CYTO_LVL'])])

pipe_cox_v1 = make_pipeline(preprocess_v1, CoxPHSurvivalAnalysis(alpha=1e-4))

fold_scores, mean_score = run_cv_cox(pipe_cox_v1, X_train, y_train, cv, scorer)
print(fold_scores)
print(mean_score)

The model without the Missing indicators shows no significant performance improve or decrease: 0.691249 ± 0.026148. We are removing the missing indicators for the rest of our study.

## 5. Feature selection with Coxnet L1

In [ ]:
preprocess_cox_net = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER', 'CYTO_LVL'])])

pipe_coxnet = make_pipeline(preprocess_cox_net, CoxnetSurvivalAnalysis())
folds = cv.split(X_train, y_train['OS_STATUS'])
y_train = loader.make_surv_target(y_train['OS_YEARS'], y_train['OS_STATUS'])
y_test = loader.make_surv_target(y_test['OS_YEARS'], y_test['OS_STATUS'])

pipe_coxnet.fit(X_train, y_train)
alphas = pipe_coxnet[-1].alphas_
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gcv = GridSearchCV(as_concordance_index_ipcw_scorer(pipe_coxnet, tau=7.0),
    param_grid={"estimator__coxnetsurvivalanalysis__alphas": [[a] for a in alphas]},
    cv=folds,          # tes mêmes 5 folds, même random_state
    error_score=0.5, # un CoxNet peut diverger sur un alpha : fold raté = 0.5, pas un crash
    n_jobs=4,
).fit(X_train, y_train)


In [ ]:
idx = gcv.best_index_
cvr = gcv.cv_results_

# 1. Le meilleur alpha
best_alpha = cvr["params"][idx]["estimator__coxnetsurvivalanalysis__alphas"][0]
print("best alpha :", best_alpha)

# 2. Les scores fold par fold au meilleur alpha  → pour le test apparié
per_fold = [cvr[f"split{i}_test_score"][idx] for i in range(5)]
print("scores fold par fold :", per_fold)

# 3. La courbe complète (pour vérifier la forme du chemin)
alphas = [p["estimator__coxnetsurvivalanalysis__alphas"][0] for p in cvr["params"]]
means  = cvr["mean_test_score"]
print("alphas :", alphas)
print("scores :", means)

In [ ]:
print(gcv.best_score_)

In [ ]:
alpha = 0.0025288732011953413
preprocess_cox_net = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ['CENTER', 'CYTO_LVL'])])

pipe_coxnet = make_pipeline(preprocess_cox_net, CoxnetSurvivalAnalysis(alphas=[alpha]))
folds = cv.split(X_train, y_train['OS_STATUS'])
y_train = loader.make_surv_target(y_train['OS_YEARS'], y_train['OS_STATUS'])
y_test = loader.make_surv_target(y_test['OS_YEARS'], y_test['OS_STATUS'])

pipe_coxnet.fit(X_train, y_train)
print(evaluation.ipcw_c_index(y_train, y_test, pipe_coxnet.predict(X_test)))
